# FarmTech Solutions — Fase 6: Visão Computacional
**Enzo Caetano Peracio Rodrigues — RM 570352** · Projeto individual

Esta entrega prepara a arquitetura, o código e o roteiro de execução.
O dataset local contém 80 fotos do Open Images e labels revisados no Make Sense.
Não há treinamentos, métricas finais ou conclusões experimentais nesta versão.

## 2. Contexto do projeto
A FarmTech Solutions explora visão computacional. Comparamos detecção de
objetos com YOLOv5 customizada e YOLOv5 padrão COCO, e classificação com uma
CNN própria. O notebook é o artefato acadêmico principal; todas as funções são definidas em células deste próprio notebook.

## 3. Objetivos
Preparar as três abordagens, estudar o efeito de 30/60 épocas na YOLO
customizada e comparar métricas adequadas a cada tarefa, tempos e facilidade
de uso. Nenhum resultado será inferido antes da execução real.

## 4. Classes escolhidas
`0: bottle`, `1: backpack`. São visualmente distintas e presentes no COCO,
permitindo usar o modelo padrão como referência sem customizá-lo.

## 5. Organização do dataset
Há 80 imagens locais: 40 de cada classe. Por classe: 32 treino, 4 validação, 4 teste.
Treino ajusta os pesos; validação seleciona os modelos; teste mede o desempenho
final, sem participar da seleção. Não reutilize imagens entre splits.

Os diretórios `data/{train,val,test}/{images,labels}` seguem o formato YOLO.
Os labels, e não os nomes dos arquivos, determinam a classe do manifesto CNN.
Imagens sem classe ou com ambas as classes são rejeitadas pela classificação.
As fotos foram coletadas do Open Images, inspecionadas visualmente e salvas
localmente. Os 80 labels (91 caixas) vieram das anotações da fonte e foram importados,
revisados visualmente e corrigidos no Make Sense por Codex via navegador,
com exportações YOLO reais. A revisão alterou 13 imagens e não corresponde
a uma revisão humana independente pelo aluno. Evidências e limitações
estão em `docs/makesense_validation.md`. Registros de origem e atribuição
estão em `data/sources.csv`. Os arquivos de fotos e labels são ignorados pelo
Git: transfira o pacote do dataset ao Drive para executar no Colab.

## 6. Preparação do ambiente
Use Google Colab (Python 3.11/3.12 compatível com as dependências) ou ambiente
local isolado. No Colab, habilite `CLONE_REPOSITORY` se o repositório ainda não
estiver disponível e `INSTALL_DEPENDENCIES` na primeira execução.
Se esta versão ainda não estiver publicada, envie o pacote `fase6_colab_preparacao.zip`
para `/content/` pelo painel Arquivos do Colab e habilite `USE_SUPPORT_ZIP`.
A instalação pode requerer reiniciar a sessão; depois execute as células
novamente. Execute os experimentos somente após integrar e validar o dataset.


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
USE_SUPPORT_ZIP = False  # alternativa ao clone: upload manual do pacote preparado
SUPPORT_ZIP = Path("/content/fase6_colab_preparacao.zip")
repo_hint = os.environ.get("FASE6_REPO")
candidates = [Path(repo_hint)] if repo_hint else [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / "fase6/config/experiment.yaml").is_file()), None)
if USE_SUPPORT_ZIP:
    destination = Path("/content/Fiap-JupyterNotebooks")
    with zipfile.ZipFile(SUPPORT_ZIP) as archive:
        members = archive.infolist()
        names = [m.filename for m in members]
        if archive.testzip() is not None or len(set(n.casefold() for n in names)) != len(names):
            raise ValueError("Pacote de apoio inválido.")
        for member in members:
            relative = Path(member.filename)
            if relative.is_absolute() or ".." in relative.parts or "\\" in member.filename or not member.filename.startswith("fase6/") or member.is_dir():
                raise ValueError("Caminho inválido no pacote de apoio.")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Links simbólicos não são permitidos.")
            target = destination / relative
            if target.exists() and target.read_bytes() != archive.read(member):
                raise FileExistsError(f"Arquivo local divergente: {target}")
        for member in members:
            target = destination / member.filename
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                target.write_bytes(archive.read(member))
    REPO_ROOT = destination
if REPO_ROOT is None and CLONE_REPOSITORY:
    REPO_ROOT = Path("/content/Fiap-JupyterNotebooks")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/EnzoCaetano015/Fiap-JupyterNotebooks.git", str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise FileNotFoundError("Use o repositório ou envie o pacote de apoio e habilite USE_SUPPORT_ZIP.")
if not (REPO_ROOT / "fase6/config/dataset_package.json").is_file():
    raise FileNotFoundError("Esta cópia ainda não contém a preparação atual. Use o pacote de apoio ou publique as mudanças antes do clone.")
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "fase6/requirements.txt")], check=True)


### Configuração e caminhos

Todas as funções do projeto estão neste notebook. As células de definições devem ser executadas na ordem apresentada, antes das células que as utilizam.

In [ ]:
"""Configuration and portable paths. No heavy framework import at module load."""
from pathlib import Path
import random
import yaml

CLASSES = {0: "bottle", 1: "backpack"}
SPLITS = ("train", "val", "test")

def find_root(start=None):
    candidate = Path(start).resolve() if start else Path(globals().get("FASE_ROOT", Path.cwd())).resolve()
    if candidate.is_file():
        candidate = candidate.parent
    for parent in (candidate, *candidate.parents):
        for root in (parent, parent / "fase6"):
            if (root / "config/experiment.yaml").is_file() and (root / "notebook").is_dir():
                return root
    raise FileNotFoundError("Raiz da Fase 6 não encontrada; informe root explicitamente.")

def load_yaml(path, required=()):
    with Path(path).open(encoding="utf-8") as stream:
        value = yaml.safe_load(stream)
    if not isinstance(value, dict):
        raise ValueError("YAML deve conter um mapeamento.")
    for key in required:
        if key not in value:
            raise ValueError(f"Chave obrigatória ausente: {key}")
    return value

def resolve_path(root, path):
    path = Path(path).expanduser()
    return path.resolve() if path.is_absolute() else (Path(root) / path).resolve()

def load_data_config(root=None, dataset_root=None):
    root = Path(root).resolve() if root else find_root()
    value = load_yaml(root / "config/data.yaml", (*SPLITS, "nc", "names"))
    if value["nc"] != 2 or value["names"] != CLASSES:
        raise ValueError("Classes devem ser exatamente 0: bottle e 1: backpack.")
    for split in SPLITS:
        if not isinstance(value[split], str) or not value[split].strip():
            raise ValueError(f"Caminho inválido para {split}.")
        value[split] = str((Path(dataset_root) / split / "images").resolve()) if dataset_root else str(resolve_path(root, value[split]))
    return value

def load_experiment(root=None):
    root = Path(root).resolve() if root else find_root()
    value = load_yaml(root / "config/experiment.yaml", ("project", "dataset", "yolo_custom", "yolo_standard", "cnn"))
    required = {"project": ("seed",), "dataset": ("classes", "expected"), "yolo_custom": ("model", "image_size", "batch_size", "epochs"), "yolo_standard": ("model",), "cnn": ("image_size", "batch_size", "epochs")}
    for section, keys in required.items():
        if not isinstance(value[section], dict):
            raise ValueError(f"Seção inválida: {section}")
        for key in keys:
            if key not in value[section]:
                raise ValueError(f"Chave obrigatória ausente: {section}.{key}")
    if value["dataset"]["classes"] != list(CLASSES.values()):
        raise ValueError("Ordem das classes inválida.")
    expected = value["dataset"]["expected"]
    if not isinstance(expected, dict):
        raise ValueError("dataset.expected deve ser um mapeamento.")
    for split in SPLITS:
        _positive_int(expected.get(f"{split}_per_class"), f"expected.{split}")
    seed = value["project"]["seed"]
    if type(seed) is not int or seed < 0:
        raise ValueError("Seed deve ser inteiro não negativo.")
    if value["yolo_custom"]["epochs"] != [30, 60]:
        raise ValueError("Experimentos obrigatórios: 30 e 60 épocas.")
    for section in ("yolo_custom", "yolo_standard"):
        if value[section]["model"] != "yolov5s":
            raise ValueError("Modelo esperado: yolov5s.")
    for section in ("yolo_custom", "cnn"):
        _positive_int(value[section]["batch_size"], f"{section}.batch_size")
    _positive_int(value["yolo_custom"]["image_size"], "yolo_custom.image_size")
    size = value["cnn"]["image_size"]
    if not isinstance(size, list) or len(size) != 2:
        raise ValueError("CNN image_size deve ter altura e largura.")
    for dimension in size:
        _positive_int(dimension, "cnn.image_size")
    _positive_int(value["cnn"]["epochs"], "cnn.epochs")
    return value

def _positive_int(value, name):
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} deve ser inteiro positivo.")

def apply_seed(seed=42, tensorflow=False, torch=False):
    random.seed(seed)
    import numpy as np
    np.random.seed(seed)
    if tensorflow:
        import tensorflow as tf
        tf.keras.utils.set_random_seed(seed)
    if torch:
        import torch as framework
        framework.manual_seed(seed)
        if framework.cuda.is_available():
            framework.cuda.manual_seed_all(seed)


In [ ]:
from pathlib import Path
import sys, os, json
FASE_ROOT = REPO_ROOT / "fase6"
CONFIG = load_experiment(FASE_ROOT)
apply_seed(CONFIG["project"]["seed"])
DATASET_ROOT = Path(os.environ.get("FASE6_DATASET", str(FASE_ROOT / "data")))
# Cada ação que instala, extrai, usa Drive ou GPU é explícita.
RUN_EXPERIMENTS = False
MOUNT_DRIVE = False
EXTRACT_DATASET = False
PREPARE_YOLO = False
INSTALL_YOLO_DEPENDENCIES = False
RUN_PREFLIGHT = False
RUN_CNN_SMOKE_TEST = False
RESTORE_RESULTS = False
EXPORT_RESULTS = False
DOWNLOAD_RESULTS = False
SESSION_NAME = "fase6_final_001"  # novo nome para um novo conjunto de experimentos
DRIVE_BASE = Path("/content/drive/MyDrive/Fiap/Fase6")
DRIVE_DATASET = DRIVE_BASE / "data"
DATASET_ZIP = DRIVE_BASE / "dataset_fase6_final.zip"
SESSION_ROOT = DRIVE_BASE / "experiments" / SESSION_NAME
RESULTS_ROOT = FASE_ROOT / "results"
YOLO_RUNTIME = None
PREFLIGHT = None
custom_runs = {}
selected = custom_test = custom_inference = standard_inference = cnn_run = cnn_metrics = None
print("Ações externas e experimentos desativados por padrão.")


## 7. Google Drive e transferência do dataset — Task 11
Você fará o upload de `dataset_fase6_final.zip` para `MyDrive/Fiap/Fase6/`.
Habilite `MOUNT_DRIVE` e, para o ZIP, `EXTRACT_DATASET`. A extração abaixo
verifica o SHA-256 e todos os membros contra o manifesto versionado,
rejeita caminhos inseguros e recusa sobrescrever uma pasta existente.
Se `data/` já foi extraída, deixe `EXTRACT_DATASET=False`.
`FASE6_DATASET` tem prioridade sobre `DRIVE_DATASET`.

Cada sessão salva runs e checkpoints em `experiments/SESSION_NAME/` no Drive.
Use o mesmo nome para recuperar resultados concluídos com `RESTORE_RESULTS`;
use outro nome para novos treinamentos. Runs existentes nunca são reiniciados
ou sobrescritos automaticamente. A escolha do upload é manual.


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_ROOT = Path(os.environ.get("FASE6_DATASET", str(DRIVE_DATASET)))
print("Dataset configurado:", DATASET_ROOT)
print("ZIP para upload manual:", DATASET_ZIP)
print("Sessão de experimentos:", SESSION_ROOT)


## 8. Validação do dataset
A inspeção funciona sem dados. A execução de modelos exige estrutura,
pares, labels e distribuição 32/4/4 válidos. A contagem é por imagem; múltiplos
objetos da mesma classe não aumentam a contagem de imagens daquela classe.

### Validação do dataset e manifesto

As funções abaixo verificam a estrutura, os labels e a distribuição por classe. O manifesto da CNN usa os labels e rejeita imagens ambíguas.

In [ ]:
"""Dataset contract; class counts refer to images, not object counts."""
from pathlib import Path
from collections import Counter
import math

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def list_images(directory):
    return sorted(p for p in Path(directory).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if Path(directory).is_dir() else []

def parse_yolo_label(path):
    rows = []
    for line_number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"{path}:{line_number}: esperado class_id x y width height.")
        try:
            class_id = int(parts[0])
            coordinates = tuple(float(x) for x in parts[1:])
        except ValueError as exc:
            raise ValueError(f"{path}:{line_number}: valores inválidos.") from exc
        if class_id not in CLASSES:
            raise ValueError(f"{path}:{line_number}: class_id deve ser 0 ou 1.")
        if any(not math.isfinite(x) or not 0 <= x <= 1 for x in coordinates) or coordinates[2] <= 0 or coordinates[3] <= 0:
            raise ValueError(f"{path}:{line_number}: coordenadas normalizadas inválidas.")
        x, y, width, height = coordinates
        if x - width/2 < -1e-5 or y - height/2 < -1e-5 or x + width/2 > 1 + 1e-5 or y + height/2 > 1 + 1e-5:
            raise ValueError(f"{path}:{line_number}: caixa ultrapassa a imagem.")
        rows.append((class_id, *coordinates))
    return rows

def inspect_dataset(data_root, expected=None):
    data_root = Path(data_root)
    expected = expected or {"train_per_class": 32, "val_per_class": 4, "test_per_class": 4}
    report = {"status": "empty", "message": "Dataset ainda não integrado.", "splits": {}, "errors": [], "total_images": 0, "total_labels": 0}
    for split in SPLITS:
        images_dir, labels_dir = data_root / split / "images", data_root / split / "labels"
        for directory in (images_dir, labels_dir):
            if not directory.is_dir():
                report["errors"].append(f"Diretório ausente: {directory}")
        images = list_images(images_dir)
        labels = sorted(labels_dir.glob("*.txt")) if labels_dir.is_dir() else []
        counts, objects = Counter(), Counter()
        ambiguous, unclassified = [], []
        stems = Counter(image.stem for image in images)
        for stem, count in stems.items():
            if count > 1:
                report["errors"].append(f"{split}: imagens com stem duplicado: {stem}")
        for image in images:
            try:
                from PIL import Image
                with Image.open(image) as decoded:
                    decoded.verify()
                with Image.open(image) as decoded:
                    decoded.convert("RGB").load()
            except (OSError, ValueError) as exc:
                report["errors"].append(f"Imagem ilegível: {image}: {exc}")
            label = labels_dir / f"{image.stem}.txt"
            if not label.is_file():
                report["errors"].append(f"Label ausente: {label}")
                continue
            try:
                rows = parse_yolo_label(label)
            except ValueError as exc:
                report["errors"].append(str(exc))
                continue
            classes = {row[0] for row in rows}
            objects.update(row[0] for row in rows)
            if len(classes) == 1:
                counts.update(classes)
            elif len(classes) > 1:
                ambiguous.append(str(image))
            else:
                unclassified.append(str(image))
        for label in labels:
            if label.stem not in stems:
                report["errors"].append(f"Label sem imagem: {label}")
                try:
                    parse_yolo_label(label)
                except ValueError as exc:
                    report["errors"].append(str(exc))
        target = expected[f"{split}_per_class"]
        report["splits"][split] = {"images": len(images), "labels": len(labels), "images_per_class": {name: counts[cid] for cid, name in CLASSES.items()}, "objects_per_class": {name: objects[cid] for cid, name in CLASSES.items()}, "ambiguous": ambiguous, "unclassified": unclassified, "expected_per_class": target, "distribution_matches": all(counts[cid] == target for cid in CLASSES) and not ambiguous and not unclassified}
        report["total_images"] += len(images)
        report["total_labels"] += len(labels)
    if report["errors"]:
        report.update(status="invalid", message="Dataset inválido; consulte errors.")
    elif report["total_images"]:
        report.update(status="ready" if all(s["distribution_matches"] for s in report["splits"].values()) else "incomplete", message="Dataset validado." if all(s["distribution_matches"] for s in report["splits"].values()) else "Distribuição acadêmica incompleta ou ambígua.")
    return report

def require_dataset(data_root, expected=None):
    report = inspect_dataset(data_root, expected)
    if report["status"] != "ready":
        details = "; ".join(report["errors"][:3])
        raise ValueError(report["message"] + (" " + details if details else ""))
    return report

def classification_manifest(data_root, splits=SPLITS):
    import pandas as pd
    records = []
    for split in splits:
        if split not in SPLITS:
            raise ValueError(f"Split inválido: {split}")
        for image in list_images(Path(data_root) / split / "images"):
            label = Path(data_root) / split / "labels" / f"{image.stem}.txt"
            if not label.is_file():
                raise ValueError(f"Imagem sem label: {image}")
            ids = {row[0] for row in parse_yolo_label(label)}
            if len(ids) != 1:
                raise ValueError(f"Imagem ambígua ou sem classe: {image}")
            cid = next(iter(ids))
            records.append({"image_path": str(image.resolve()), "class_id": cid, "class_name": CLASSES[cid], "split": split})
    return pd.DataFrame(records, columns=["image_path", "class_id", "class_name", "split"])


In [ ]:
# Transferência verificada e identificação do dataset, sem frameworks pesados.
import hashlib, json, shutil, tempfile, zipfile
from datetime import datetime, timezone

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def dataset_fingerprint(data_root):
    root = Path(data_root)
    records = []
    for split in SPLITS:
        for image in list_images(root / split / "images"):
            label = root / split / "labels" / f"{image.stem}.txt"
            for path in (image, label):
                records.append([path.relative_to(root).as_posix(), file_sha256(path)])
    return hashlib.sha256(json.dumps(sorted(records), separators=(",", ":")).encode()).hexdigest()

def extract_dataset_zip(archive_path, destination, package_manifest):
    archive_path, destination = Path(archive_path), Path(destination)
    if destination.exists():
        raise FileExistsError(f"Dataset já existe: {destination}; não será sobrescrito.")
    if file_sha256(archive_path) != package_manifest["sha256"]:
        raise ValueError("SHA-256 do ZIP diferente do pacote revisado; não extrair.")
    expected = {row["path"]: row for row in package_manifest["files"]}
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("ZIP corrompido.")
        members = archive.infolist()
        names = [member.filename for member in members]
        if len(set(n.casefold() for n in names)) != len(names) or set(names) != set(expected):
            raise ValueError("Membros duplicados ou diferentes do manifesto.")
        for member in members:
            path = Path(member.filename)
            if path.is_absolute() or ".." in path.parts or "\\" in member.filename or not member.filename.startswith("data/"):
                raise ValueError("Caminho inseguro no ZIP.")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Links simbólicos não são permitidos.")
            raw = archive.read(member)
            metadata = expected[member.filename]
            if len(raw) != metadata["bytes"] or hashlib.sha256(raw).hexdigest() != metadata["sha256"]:
                raise ValueError(f"Arquivo diferente do manifesto: {member.filename}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=destination.parent) as temp:
            stage = Path(temp)
            for member in members:
                path = stage / member.filename
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(archive.read(member))
            require_dataset(stage / "data")
            shutil.move(str(stage / "data"), str(destination))
    return destination

PACKAGE_MANIFEST = json.loads((FASE_ROOT / "config/dataset_package.json").read_text(encoding="utf-8"))
if EXTRACT_DATASET:
    if not MOUNT_DRIVE:
        raise RuntimeError("Monte o Drive antes de extrair o ZIP.")
    extract_dataset_zip(DATASET_ZIP, DRIVE_DATASET, PACKAGE_MANIFEST)
    DATASET_ROOT = Path(os.environ.get("FASE6_DATASET", str(DRIVE_DATASET)))


In [ ]:
DATA_CONFIG = load_data_config(FASE_ROOT, DATASET_ROOT)
dataset_report = inspect_dataset(DATASET_ROOT, CONFIG["dataset"]["expected"])
DATASET_READY = dataset_report["status"] == "ready"
print("Diagnóstico de", DATASET_ROOT)
print(json.dumps(dataset_report, indent=2, ensure_ascii=False))
if DATASET_READY:
    DATASET_FINGERPRINT = dataset_fingerprint(DATASET_ROOT)
    if DATASET_FINGERPRINT != PACKAGE_MANIFEST["dataset_fingerprint"]:
        DATASET_READY = False
        raise ValueError("Dataset diferente da revisão Make Sense aprovada; interrompido.")
    print("Dataset confirmado:", DATASET_FINGERPRINT)


## 9. YOLOv5 customizada
A customizada usa arquitetura `yolov5s` e pesos aleatórios, sem Transfer
Learning. A YOLO padrão usa pesos COCO e não é treinada neste projeto.
O runtime oficial está fixado em `v7.0`; os dois treinos usam seed 42,
imagem 640 e batch 16. A parada antecipada está desativada.

30 e 60 épocas permitem estudar a duração do treino. Os outros parâmetros
ficam iguais, embora o agendamento interno de learning rate do YOLO dependa
do número total de épocas. Mantenha o mesmo hardware e registre a sessão.
Precision e recall medem detecções; mAP agrega precisão nos limiares de IoU.
Não existe uma accuracy de detecção equivalente à accuracy da CNN.

**Células a seguir dependem do dataset real e da habilitação de `RUN_EXPERIMENTS`.**

### Métricas e cronometragem

Utilitários compartilhados pelos experimentos: tempos, tabelas e gráficos. Valores ausentes permanecem indisponíveis, e gráficos vazios são rejeitados.

In [ ]:
"""Shared inference timing; loading and warm-up belong outside measured calls."""
from time import perf_counter
import statistics
import math

def measure_call(function, *args, synchronize=None, **kwargs):
    if synchronize:
        synchronize()
    start = perf_counter()
    result = function(*args, **kwargs)
    if synchronize:
        synchronize()
    return result, (perf_counter() - start) * 1000

def summarize_times(milliseconds):
    values = list(milliseconds)
    if not values:
        return {"count": 0, "mean_inference_ms": None, "median_inference_ms": None, "std_inference_ms": None}
    if any(not isinstance(x, (int, float)) or not math.isfinite(x) or x < 0 for x in values):
        raise ValueError("Tempos devem ser números finitos não negativos.")
    return {"count": len(values), "mean_inference_ms": statistics.mean(values), "median_inference_ms": statistics.median(values), "std_inference_ms": statistics.stdev(values) if len(values) > 1 else None}


YOLO_COLUMNS = ["experiment", "epochs", "best_epoch", "precision", "recall", "map50", "map50_95", "box_loss", "obj_loss", "cls_loss", "training_seconds"]
BENCHMARK_COLUMNS = ["approach", "task_type", "training_seconds", "mean_inference_ms", "median_inference_ms", "std_inference_ms", "primary_metric", "primary_metric_value", "precision", "recall", "f1", "map50", "map50_95", "notes"]

def compare_yolo_epochs(records):
    import pandas as pd
    if not records:
        return pd.DataFrame(columns=YOLO_COLUMNS)
    table = pd.DataFrame(records)
    validate_columns(table, ["experiment", "epochs"])
    validate_numbers(table, YOLO_COLUMNS[1:])
    if any(epoch not in (30, 60) for epoch in table.epochs):
        raise ValueError("Comparação espera experimentos de 30/60 épocas.")
    return table.reindex(columns=YOLO_COLUMNS).sort_values("epochs").reset_index(drop=True)

def select_custom_experiment(records):
    import pandas as pd
    table = pd.DataFrame(records)
    if table.empty or "metrics_split" not in table or any(table.metrics_split != "val"):
        raise ValueError("Seleção deve usar exclusivamente métricas de validação.")
    for column in ("map50_95", "map50", "epochs"):
        if column not in table:
            raise ValueError(f"Métrica necessária para seleção: {column}")
    validate_numbers(table, ["map50_95", "map50", "epochs"])
    if table[["map50_95", "map50"]].isna().any().any():
        raise ValueError("Seleção exige mAP disponível.")
    validate_numbers(table, ["training_seconds"])
    if "training_seconds" not in table or table.training_seconds.isna().any() or (table.training_seconds <= 0).any():
        raise ValueError("Seleção exige duração real positiva para o desempate.")
    index = table.sort_values(["map50_95", "map50", "training_seconds", "epochs"], ascending=[False, False, True, True]).index[0]
    return records[int(index)]

def build_benchmark(records):
    import pandas as pd
    if not records:
        return pd.DataFrame(columns=BENCHMARK_COLUMNS)
    table = pd.DataFrame(records)
    validate_columns(table, BENCHMARK_COLUMNS)
    validate_numbers(table, ["training_seconds", "mean_inference_ms", "median_inference_ms", "std_inference_ms", "primary_metric_value", "precision", "recall", "f1", "map50", "map50_95"])
    if table.approach.duplicated().any():
        raise ValueError("Uma linha por abordagem no benchmark geral.")
    for row in table.itertuples():
        if row.approach not in ("yolo_custom", "yolo_standard", "cnn"):
            raise ValueError("Abordagem desconhecida.")
        if pd.isna(row.primary_metric) != pd.isna(row.primary_metric_value):
            raise ValueError("Nome e valor da métrica devem estar presentes juntos.")
        allowed = ("accuracy", "f1") if row.approach == "cnn" else ("map50", "map50_95")
        if pd.notna(row.primary_metric) and row.primary_metric not in allowed:
            raise ValueError("Não misturar métricas de classificação e detecção.")
        if row.task_type != ("classification" if row.approach == "cnn" else "detection"):
            raise ValueError("Tipo de tarefa incoerente.")
        for value in (row.training_seconds, row.mean_inference_ms, row.median_inference_ms, row.std_inference_ms):
            if pd.notna(value) and value < 0:
                raise ValueError("Tempo não pode ser negativo.")
    return table.reindex(columns=BENCHMARK_COLUMNS)


In [ ]:
"""Load, validate and visualize real metrics; missing values stay missing."""
from pathlib import Path
import json
import math
import pandas as pd

def validate_columns(table, required):
    missing = set(required) - set(table.columns)
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing)}")
    return table

def load_metrics(path, required=()):
    path = Path(path)
    if path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, (dict, list)):
            raise ValueError("JSON de métricas deve ser objeto ou lista de objetos.")
        table = pd.DataFrame([data] if isinstance(data, dict) else data)
    elif path.suffix.lower() == ".csv":
        table = pd.read_csv(path)
        table.columns = table.columns.str.strip()
    else:
        raise ValueError("Use JSON ou CSV.")
    return validate_columns(table, required)

def validate_numbers(table, columns):
    for column in columns:
        if column not in table:
            continue
        for value in table[column].dropna():
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"Número inválido em {column}: {value}")
    return table

def format_number(value, digits=3):
    return "Indisponível" if value is None or pd.isna(value) else f"{value:.{digits}f}"

def save_table(table, output):
    if table.empty:
        raise ValueError("Não salvar tabela vazia como resultado.")
    path = Path(output)
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    return path

def _save_figure(figure, output):
    import matplotlib.pyplot as plt
    path = Path(output)
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path

def plot_yolo_curves(csv_path, output):
    import matplotlib.pyplot as plt
    table = load_metrics(csv_path, ["epoch"])
    performance = [x for x in ("metrics/precision", "metrics/recall", "metrics/mAP_0.5", "metrics/mAP_0.5:0.95") if x in table and table[x].notna().any()]
    losses = [x for x in table.columns if x.startswith(("train/", "val/")) and x.endswith("loss")]
    if table.empty or not performance:
        raise ValueError("Sem métricas YOLO para gerar gráfico.")
    validate_numbers(table, ["epoch", *performance, *losses])
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    for column in performance:
        axes[0].plot(table.epoch + 1, table[column], label=column)
    for column in losses:
        axes[1].plot(table.epoch + 1, table[column], label=column)
    for ax, ylabel in zip(axes, ("Validação", "Loss treino/validação")):
        ax.set(xlabel="Época", ylabel=ylabel); ax.legend(fontsize=7)
    return _save_figure(figure, output)


def plot_cnn_history(history, output_dir):
    import matplotlib.pyplot as plt
    if not history or not history.get("loss") or not history.get("accuracy"):
        raise ValueError("Histórico CNN vazio ou incompleto.")
    files = []
    for metric in ("loss", "accuracy"):
        figure, ax = plt.subplots()
        for key in (metric, f"val_{metric}"):
            if history.get(key):
                ax.plot(range(1, len(history[key]) + 1), history[key], label=key)
        ax.set(xlabel="Época", ylabel=metric)
        ax.legend()
        files.append(_save_figure(figure, Path(output_dir) / f"cnn_{metric}.png"))
    return files

def plot_confusion_matrix(metrics, output):
    import numpy as np
    import matplotlib.pyplot as plt
    matrix = np.asarray(metrics.get("confusion_matrix", []))
    if matrix.shape != (2, 2) or matrix.sum() == 0:
        raise ValueError("Matriz de confusão vazia ou inválida.")
    figure, ax = plt.subplots()
    ax.imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
    names = metrics.get("class_names", ["bottle", "backpack"])
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=names, yticklabels=names, xlabel="Predição", ylabel="Classe real")
    return _save_figure(figure, output)

def plot_times(table, output):
    import matplotlib.pyplot as plt
    validate_columns(table, ["approach", "mean_inference_ms"])
    available = table.dropna(subset=["mean_inference_ms"])
    if available.empty:
        raise ValueError("Sem tempos reais para gerar gráfico.")
    validate_numbers(available, ["mean_inference_ms"])
    figure, ax = plt.subplots()
    ax.bar(available.approach, available.mean_inference_ms)
    ax.set(ylabel="Tempo médio por imagem (ms)")
    return _save_figure(figure, output)


### Pipeline YOLOv5 customizada

Preparação do runtime oficial, comandos de treino, coleta de métricas e avaliação. Definir estas funções não baixa pesos nem inicia experimentos.

In [ ]:
"""Official YOLOv5 adapters. All downloads and executions are explicit."""
from pathlib import Path
import json
import subprocess
import sys
from time import perf_counter
import yaml

YOLO_REF = "v7.0"
YOLO_REPOSITORY = "https://github.com/ultralytics/yolov5.git"

def prepare_yolov5(runtime_dir=None, install=False):
    runtime = Path(runtime_dir or (find_root() / ".runtime/yolov5")).resolve()
    os.environ["WANDB_MODE"] = "disabled"
    os.environ["COMET_MODE"] = "DISABLED"
    if not runtime.exists():
        runtime.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "--branch", YOLO_REF, YOLO_REPOSITORY, str(runtime)], check=True)
    for script in ("train.py", "val.py", "hubconf.py", "models/yolov5s.yaml", "data/hyps/hyp.scratch-low.yaml"):
        if not (runtime / script).is_file():
            raise ValueError(f"Runtime YOLOv5 inválido: {runtime}")
    ref = subprocess.run(["git", "describe", "--tags", "--exact-match", "HEAD"], cwd=runtime, capture_output=True, text=True, check=True).stdout.strip()
    if ref != YOLO_REF:
        raise ValueError(f"Runtime deve estar na revisão {YOLO_REF}, recebido {ref}.")
    if install:
        # v7.0 uses trusted official checkpoints with the pre-2.6 loading API.
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "torchvision==0.20.1", "numpy>=1.26,<2", "setuptools>=65,<81"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(runtime / "requirements.txt"), "torch==2.5.1", "torchvision==0.20.1", "numpy>=1.26,<2", "setuptools>=65,<81"], check=True)
    return runtime

def write_runtime_data(root=None, dataset_root=None, output=None):
    root = Path(root).resolve() if root else find_root()
    value = load_data_config(root, dataset_root)
    output = Path(output or root / ".runtime/data.yaml")
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(yaml.safe_dump(value, sort_keys=False), encoding="utf-8")
    return output.resolve()

def build_train_command(runtime, data_yaml, epochs, batch_size=16, image_size=640, seed=42, output_dir=None, model="yolov5s"):
    if epochs not in (30, 60) or model != "yolov5s":
        raise ValueError("Use yolov5s e 30 ou 60 épocas.")
    if batch_size <= 0 or image_size <= 0:
        raise ValueError("Batch e tamanho de imagem devem ser positivos.")
    runtime = Path(runtime).resolve()
    output_dir = Path(output_dir or find_root() / "runs/yolo_custom").resolve()
    return [sys.executable, str(runtime / "train.py"), "--data", str(Path(data_yaml).resolve()), "--cfg", str(runtime / f"models/{model}.yaml"), "--weights", "", "--epochs", str(epochs), "--batch-size", str(batch_size), "--imgsz", str(image_size), "--seed", str(seed), "--patience", "0", "--noplots", "--device", "0", "--workers", "2", "--hyp", str(runtime / "data/hyps/hyp.scratch-low.yaml"), "--project", str(output_dir), "--name", f"custom_{epochs}_epochs"]

def locate_best_weights(run_dir):
    path = Path(run_dir) / "weights/best.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Pesos não encontrados: {path}; execute treinamento real primeiro.")
    return path.resolve()

def collect_training_metrics(run_dir, training_seconds=None, epochs=None, output=None):
    import pandas as pd
    table = pd.read_csv(Path(run_dir) / "results.csv")
    table.columns = table.columns.str.strip()
    if table.empty:
        raise ValueError("results.csv não contém épocas executadas.")
    aliases = {"precision": "metrics/precision", "recall": "metrics/recall", "map50": "metrics/mAP_0.5", "map50_95": "metrics/mAP_0.5:0.95", "box_loss": "val/box_loss", "obj_loss": "val/obj_loss", "cls_loss": "val/cls_loss"}
    # YOLOv5 chooses best.pt by fitness = 0.1*mAP50 + 0.9*mAP50_95.
    if all(c in table for c in ("metrics/mAP_0.5", "metrics/mAP_0.5:0.95")):
        fitness = 0.1 * table["metrics/mAP_0.5"] + 0.9 * table["metrics/mAP_0.5:0.95"]
        if fitness.isna().all():
            raise ValueError("Métricas de fitness indisponíveis no CSV.")
        index = fitness[fitness == fitness.max()].index[-1]
        row = table.loc[index]
    else:
        row = table.iloc[-1]
    marker = Path(run_dir) / "best_epoch.json"
    best_epoch_source = "results_csv_rounded_fitness"
    if marker.is_file():
        recorded = json.loads(marker.read_text(encoding="utf-8"))
        best_epoch = recorded["best_epoch"]
        if type(best_epoch) is not int or not 1 <= best_epoch <= len(table) or not np.isfinite(recorded["fitness"]):
            raise ValueError("Registro da melhor época inválido.")
        row = table.iloc[best_epoch - 1]
        best_epoch_source = "official_on_model_save_callback"
    if epochs is not None and len(table) != epochs:
        raise ValueError(f"Treino incompleto: {len(table)} de {epochs} épocas.")
    if "epoch" not in table or table.epoch.astype(int).tolist() != list(range(len(table))):
        raise ValueError("Sequência de épocas inválida no results.csv.")
    result = {"experiment": Path(run_dir).name, "epochs": epochs if epochs is not None else len(table), "completed_epochs": len(table), "training_seconds": training_seconds, "metrics_split": "val", "best_epoch": int(row["epoch"]) + 1 if "epoch" in row else None}
    result["best_epoch_source"] = best_epoch_source
    for target, source in aliases.items():
        result[target] = float(row[source]) if source in row and pd.notna(row[source]) else None
    if output:
        path = Path(output)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")
    return result

def train_custom_yolo(runtime, dataset_root, root=None, epochs=30, batch_size=16, image_size=640, seed=42, output_dir=None):
    require_experiment_preflight(dataset_root)
    root = Path(root).resolve() if root else find_root()
    data_yaml = write_runtime_data(root, dataset_root)
    output_dir = Path(output_dir or root / "runs/yolo_custom").resolve()
    run_dir = output_dir / f"custom_{epochs}_epochs"
    if run_dir.exists():
        raise FileExistsError(f"Experimento já existe: {run_dir}; escolha outro output_dir.")
    command = build_train_command(runtime, data_yaml, epochs, batch_size, image_size, seed, output_dir)
    start = perf_counter()
    worker = 'import json\nfrom pathlib import Path\nfrom train import parse_opt, main\nfrom utils.callbacks import Callbacks\n\ndef remember_best(last, epoch, final_epoch, best_fitness, fitness):\n    if best_fitness == fitness:\n        marker = Path(last).parent.parent / "best_epoch.json"\n        marker.write_text(json.dumps({"best_epoch": int(epoch) + 1, "fitness": float(fitness)}), encoding="utf-8")\n\ncallbacks = Callbacks()\ncallbacks.register_action("on_model_save", name="fase6_best_epoch", callback=remember_best)\nmain(parse_opt(), callbacks=callbacks)\n'
    subprocess.run([sys.executable, "-c", worker, *command[2:]], cwd=Path(runtime), check=True)
    duration = perf_counter() - start
    if not (run_dir / "best_epoch.json").is_file():
        raise RuntimeError("YOLOv5 não registrou a época do checkpoint pelo callback oficial.")
    metrics = collect_training_metrics(run_dir, duration, epochs)
    metrics.update(configuration={"model": "yolov5s", "weights": "", "image_size": image_size, "batch_size": batch_size, "seed": seed, "patience": 0, "optimizer": "SGD", "workers": 2, "hyperparameters": load_yaml(run_dir / "hyp.yaml")}, dataset_fingerprint=dataset_fingerprint(dataset_root), runtime_sha=PREFLIGHT["yolov5_commit"], environment=PREFLIGHT, checkpoint_sha256=file_sha256(locate_best_weights(run_dir)))
    save_json(metrics, root / f"results/yolo_custom/metrics_{epochs}_epochs.json")
    shutil.copy2(run_dir / "results.csv", root / f"results/yolo_custom/history_{epochs}_epochs.csv")
    return {"run_dir": run_dir, "weights": locate_best_weights(run_dir), "metrics": metrics}

def validate_custom_yolo(runtime, weights, dataset_root, root=None, split="test", image_size=640, batch_size=16, output_dir=None):
    require_experiment_preflight(dataset_root)
    if split not in ("val", "test"):
        raise ValueError("Avaliação deve usar val ou test.")
    if not Path(weights).is_file():
        raise FileNotFoundError(weights)
    root = Path(root).resolve() if root else find_root()
    data_yaml = write_runtime_data(root, dataset_root)
    output_dir = Path(output_dir or root / "runs/yolo_evaluation")
    name = f"{Path(weights).parents[1].name}_{split}"
    if (output_dir / name).exists():
        raise FileExistsError(f"Avaliação já existe: {output_dir / name}")
    # Official val.run returns (metrics, per-class maps, timing).
    code = 'import json, sys\nfrom pathlib import Path\nimport torch\nfrom val import run\nfrom models.experimental import attempt_load\nfrom utils.general import check_dataset\nfrom utils.dataloaders import create_dataloader\nfrom utils.loss import ComputeLoss\n\ndata = check_dataset(sys.argv[1])\nmodel = attempt_load(sys.argv[2], device=torch.device("cuda:0"), fuse=False).float()\nif not hasattr(model, "hyp"):\n    raise RuntimeError("Checkpoint não contém os hiperparâmetros necessários para calcular losses.")\nloss = ComputeLoss(model)\nimgsz, batch = int(sys.argv[4]), int(sys.argv[5])\nloader = create_dataloader(data[sys.argv[3]], imgsz, batch, int(model.stride.max()), single_cls=False, pad=0.5, rect=True, workers=2)[0]\nsave_dir = Path(sys.argv[6]) / sys.argv[7]\nsave_dir.mkdir(parents=True, exist_ok=False)\nr, maps, timings = run(data=data, weights=sys.argv[2], task=sys.argv[3], imgsz=imgsz, batch_size=batch, model=model, dataloader=loader, save_dir=save_dir, compute_loss=loss, half=False, plots=False)\nprint("FASE6_METRICS=" + json.dumps({"precision":float(r[0]),"recall":float(r[1]),"map50":float(r[2]),"map50_95":float(r[3]),"box_loss":float(r[4]),"obj_loss":float(r[5]),"cls_loss":float(r[6]),"metrics_split":sys.argv[3],"loss_computed":True,"evaluation_precision":"FP32","images_evaluated":len(loader.dataset)}))\n'
    result = subprocess.run([sys.executable, "-c", code, str(data_yaml), str(Path(weights).resolve()), split, str(image_size), str(batch_size), str(output_dir), name], cwd=Path(runtime), text=True, capture_output=True, check=True)
    lines = [line for line in result.stdout.splitlines() if line.startswith("FASE6_METRICS=")]
    if not lines:
        raise RuntimeError("YOLOv5 não retornou métricas estruturadas.")
    metrics = json.loads(lines[-1].split("=", 1)[1])
    if metrics["images_evaluated"] != 8:
        raise ValueError("Avaliação não usou as oito imagens do split reservado.")
    return metrics

def detect_with_custom_yolo(runtime, weights, dataset_root, output_dir, image_size=640, warmup=1):
    require_experiment_preflight(dataset_root)
    if not Path(weights).is_file():
        raise FileNotFoundError(weights)
    import torch
    model = torch.hub.load(str(Path(runtime).resolve()), "custom", path=str(Path(weights).resolve()), source="local", autoshape=True)
    model = model.to("cuda:0").eval()
    return infer_folder(model, Path(dataset_root) / "test/images", output_dir=output_dir, image_size=image_size, warmup=warmup)


### Inferência comum às duas YOLOs

As funções abaixo carregam o modelo COCO, resolvem classes por nome e realizam inferência com tabelas e tempos por imagem. A YOLO customizada também utiliza estes utilitários; por isso são definidos antes dos experimentos.

In [ ]:
"""COCO baseline and common detection inference, with IDs resolved by name."""
from pathlib import Path
import json
import sys
import pandas as pd

DETECTION_COLUMNS = ["image", "predicted_class", "confidence", "xmin", "ymin", "xmax", "ymax", "inference_ms"]

def resolve_class_ids(names, requested=tuple(CLASSES.values())):
    mapping = dict(enumerate(names)) if isinstance(names, (list, tuple)) else dict(names)
    reverse = {name: int(cid) for cid, name in mapping.items()}
    missing = set(requested) - reverse.keys()
    if missing:
        raise ValueError(f"Classes ausentes no modelo: {sorted(missing)}")
    return {name: reverse[name] for name in requested}

def load_standard_yolo(runtime, model_name="yolov5s"):
    if model_name != "yolov5s":
        raise ValueError("Modelo esperado: yolov5s.")
    import torch
    model = torch.hub.load(str(Path(runtime).resolve()), model_name, source="local", pretrained=True, autoshape=True)
    model.classes = list(resolve_class_ids(model.names).values())
    model = model.to("cuda:0").eval()
    return model

def normalize_detections(rows, names, image, inference_ms):
    ids = resolve_class_ids(names)
    selected = {cid: name for name, cid in ids.items()}
    records = []
    for row in rows:
        if len(row) < 6:
            raise ValueError("Detecção deve conter xyxy, confiança e class_id.")
        xmin, ymin, xmax, ymax, confidence, cid = row[:6]
        if int(cid) not in selected:
            continue
        records.append({"image": str(image), "predicted_class": selected[int(cid)], "confidence": float(confidence), "xmin": float(xmin), "ymin": float(ymin), "xmax": float(xmax), "ymax": float(ymax), "inference_ms": inference_ms})
    # Keep an explicit no-detection row; timings never depend on object count.
    if not records:
        records.append({"image": str(image), "predicted_class": None, "confidence": None, "xmin": None, "ymin": None, "xmax": None, "ymax": None, "inference_ms": inference_ms})
    return pd.DataFrame(records, columns=DETECTION_COLUMNS)

def _synchronizer(model):
    torch = sys.modules.get("torch")
    if torch is None:
        return None
    try:
        device = next(model.parameters()).device
        return (lambda: torch.cuda.synchronize(device)) if device.type == "cuda" else None
    except (AttributeError, StopIteration):
        return None

def infer_image(model, image_path, image_size=640):
    from PIL import Image
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    result, milliseconds = measure_call(model, image, size=image_size, synchronize=_synchronizer(model))
    tensor = result.xyxy[0]
    rows = tensor.detach().cpu().tolist() if hasattr(tensor, "detach") else tensor
    return normalize_detections(rows, model.names, Path(image_path).name, milliseconds)

def infer_folder(model, images_dir, output_dir=None, image_size=640, warmup=1, max_evidence=8):
    from PIL import Image, ImageDraw
    images = list_images(images_dir)
    if not images:
        raise ValueError("Dataset ainda não integrado: nenhuma imagem de teste.")
    if warmup < 0:
        raise ValueError("warmup deve ser não negativo.")
    if hasattr(model, "eval"):
        model.eval()
    with Image.open(images[0]) as source:
        warmup_image = source.convert("RGB")
    for _ in range(warmup):
        model(warmup_image, size=image_size)
    frames, timing_rows, evidence = [], [], []
    for index, image in enumerate(images):
        table = infer_image(model, image, image_size)
        frames.append(table)
        timing_rows.append({"image": image.name, "inference_ms": float(table.inference_ms.iloc[0])})
        if output_dir and index < max_evidence:
            directory = Path(output_dir) / "detections"
            directory.mkdir(parents=True, exist_ok=True)
            with Image.open(image) as source:
                rendered = source.convert("RGB")
            draw = ImageDraw.Draw(rendered)
            for row in table.itertuples():
                if row.predicted_class is not None:
                    draw.rectangle([row.xmin, row.ymin, row.xmax, row.ymax], outline="red", width=2)
                    draw.text((row.xmin, row.ymin), f"{row.predicted_class}: {row.confidence:.2f}", fill="red")
            destination = directory / f"evidence_{image.stem}.jpg"
            rendered.thumbnail((960, 960))
            rendered.save(destination, quality=85)
            evidence.append(str(destination.resolve()))
    detections = pd.concat(frames, ignore_index=True)
    timings = pd.DataFrame(timing_rows)
    summary = summarize_times(timings.inference_ms.tolist())
    summary.update(dataset_fingerprint=globals().get("DATASET_FINGERPRINT"), environment=globals().get("PREFLIGHT"), image_size=image_size, warmup=warmup, timing_scope="model call including preprocessing and NMS; excludes image loading, model loading, warm-up and saving")
    if output_dir:
        save_inference(detections, timings, summary, output_dir)
    return {"detections": detections, "timings": timings, "summary": summary, "images": evidence}

def save_inference(detections, timings, summary, output_dir):
    if detections.empty or timings.empty:
        raise ValueError("Não salvar resultados vazios.")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    detections.to_csv(output_dir / "detections.csv", index=False)
    timings.to_csv(output_dir / "inference_times.csv", index=False)
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")


### Preflight Colab/GPU — Task 12
Primeiro habilite `PREPARE_YOLO`, `INSTALL_YOLO_DEPENDENCIES` e `RUN_PREFLIGHT`,
mantendo `RUN_EXPERIMENTS=False`. Se o instalador pedir reinício, reinicie
antes de importar os frameworks e reexecute a preparação com instalação desativada.
O preflight exige GPU nos dois frameworks e salva `environment.json` no Drive.
Só depois habilite `RUN_EXPERIMENTS` e execute os experimentos abaixo.
Em uma nova conexão, use `RESTORE_RESULTS=True` para recuperar as etapas
concluídas, mantendo dataset, hardware e versões iguais. Runs interrompidos
não têm recuperação automática: preserve-os e use uma nova sessão para repetir
os dois experimentos. [Roteiro completo](../docs/colab_execution.md).


In [ ]:
# Preflight real, preservação no Drive e recuperação de etapas concluídas.
import platform, re, gc

def save_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return path

def collect_environment(runtime, seed=42):
    import torch
    import tensorflow as tf
    devices = tf.config.list_physical_devices("GPU")
    if not torch.cuda.is_available() or not devices:
        raise RuntimeError("GPU indisponível no PyTorch ou TensorFlow. Selecione GPU no Colab; parâmetros não serão reduzidos.")
    for device in devices:
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except RuntimeError as exc:
            raise RuntimeError("Reinicie a sessão: TensorFlow já inicializou a GPU antes do preflight.") from exc
    # Minimal real GPU operations, including a convolution, before final training.
    torch.ones(1, device="cuda:0").sum().item()
    torch.cuda.synchronize()
    with tf.device("/GPU:0"):
        probe = tf.nn.conv2d(tf.zeros((1, 8, 8, 3)), tf.ones((3, 3, 3, 1)), strides=1, padding="SAME")
    if "GPU" not in probe.device:
        raise RuntimeError("TensorFlow não executou o smoke test na GPU.")
    probe.numpy()
    del probe
    commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=runtime, check=True, capture_output=True, text=True).stdout.strip()
    return {"python": platform.python_version(), "platform": platform.platform(), "torch": torch.__version__, "tensorflow": tf.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "tensorflow_gpu_count": len(devices), "gpu_smoke_test": "torch operation and TensorFlow convolution passed", "timestamp_utc": datetime.now(timezone.utc).isoformat(), "seed": seed, "yolov5_ref": YOLO_REF, "yolov5_commit": commit}

def experiment_contract(environment, dataset_root):
    return {"dataset_fingerprint": dataset_fingerprint(dataset_root), "gpu": environment["gpu"], "cuda": environment["cuda"], "torch": environment["torch"], "tensorflow": environment["tensorflow"], "python": environment["python"], "yolov5_commit": environment["yolov5_commit"], "seed": 42, "model": "yolov5s", "image_size": 640, "batch_size": 16, "patience": 0, "weights": ""}

def require_experiment_preflight(dataset_root):
    require_dataset(dataset_root)
    if not PREFLIGHT or not PREFLIGHT.get("contract"):
        raise RuntimeError("Execute o preflight com GPU antes dos experimentos.")
    if dataset_fingerprint(dataset_root) != PREFLIGHT["contract"]["dataset_fingerprint"]:
        raise ValueError("Dataset mudou após o preflight.")
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU deixou de estar disponível.")
    return PREFLIGHT

def persist_results():
    if not PREFLIGHT:
        raise RuntimeError("Sessão de execução não preparada.")
    source, destination = RESULTS_ROOT, SESSION_ROOT / "results"
    destination.mkdir(parents=True, exist_ok=True)
    for path in source.rglob("*"):
        if path.is_file():
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
    return destination

def load_saved_inference(directory):
    import pandas as pd
    directory = Path(directory)
    data = {"detections": pd.read_csv(directory / "detections.csv"), "timings": pd.read_csv(directory / "inference_times.csv"), "summary": json.loads((directory / "summary.json").read_text(encoding="utf-8")), "images": [str(p) for p in sorted((directory / "detections").glob("*.jpg"))]}
    expected = {p.name for p in list_images(DATASET_ROOT / "test/images")}
    if set(data["timings"].image) != expected or len(data["timings"]) != 8:
        raise ValueError("Inferência preservada não usa as mesmas oito imagens de teste.")
    if data["summary"].get("dataset_fingerprint") != DATASET_FINGERPRINT:
        raise ValueError("Inferência preservada pertence a outro dataset.")
    validate_numbers(data["timings"], ["inference_ms"])
    if data["timings"].inference_ms.isna().any() or (data["timings"].inference_ms < 0).any():
        raise ValueError("Tempos preservados inválidos.")
    if data["summary"].get("count") != 8 or len(data["images"]) != 8:
        raise ValueError("Resumo ou evidências de inferência incompletos.")
    return data

def recover_yolo_runs():
    recovered = {}
    for epoch in (30, 60):
        metrics_file = RESULTS_ROOT / f"yolo_custom/metrics_{epoch}_epochs.json"
        if metrics_file.is_file():
            metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
            run_dir = SESSION_ROOT / f"runs/yolo_custom/custom_{epoch}_epochs"
            weights = locate_best_weights(run_dir)
            if metrics["completed_epochs"] != epoch or metrics["dataset_fingerprint"] != DATASET_FINGERPRINT or file_sha256(weights) != metrics["checkpoint_sha256"]:
                raise ValueError("Experimento preservado inválido ou divergente.")
            recovered[epoch] = {"run_dir": run_dir, "weights": weights, "metrics": metrics}
    return recovered

def preflight_colab():
    if not MOUNT_DRIVE or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Monte seu Drive antes do preflight e da preservação de checkpoints.")
    if not re.fullmatch(r"[A-Za-z0-9_-]+", SESSION_NAME):
        raise ValueError("SESSION_NAME deve usar letras, números, _ ou -.")
    report = require_dataset(DATASET_ROOT, CONFIG["dataset"]["expected"])
    expected = {"model": "yolov5s", "image_size": 640, "batch_size": 16, "epochs": [30, 60]}
    if CONFIG["yolo_custom"] != expected or CONFIG["project"]["seed"] != 42 or CONFIG["cnn"] != {"image_size": [224, 224], "batch_size": 8, "epochs": 30}:
        raise ValueError("Parâmetros finais diferentes dos aprovados.")
    if YOLO_RUNTIME is None:
        raise RuntimeError("Prepare o runtime YOLO antes do preflight.")
    runtime = prepare_yolov5(YOLO_RUNTIME, install=False)
    environment = collect_environment(runtime)
    environment["contract"] = experiment_contract(environment, DATASET_ROOT)
    environment["dataset_counts"] = {s: report["splits"][s]["images"] for s in SPLITS}
    environment["dataset_fingerprint"] = DATASET_FINGERPRINT
    existing = SESSION_ROOT / "results/environment.json"
    if SESSION_ROOT.exists():
        if not RESTORE_RESULTS or not existing.is_file():
            raise FileExistsError("Sessão já existe. Para recuperar, use RESTORE_RESULTS=True; para novo treino, use outro SESSION_NAME e uma cópia limpa do repositório.")
        old = json.loads(existing.read_text(encoding="utf-8"))
        if old["contract"] != environment["contract"]:
            raise ValueError("Hardware, bibliotecas, configuração ou dataset mudaram. Não misturar experimentos na mesma sessão.")
        for path in (SESSION_ROOT / "results").rglob("*"):
            if path.is_file():
                target = RESULTS_ROOT / path.relative_to(SESSION_ROOT / "results")
                if target.exists() and file_sha256(target) != file_sha256(path):
                    raise FileExistsError("Resultados locais divergentes; recupere em uma cópia limpa do repositório.")
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
        environment = old
    else:
        if any(RESULTS_ROOT.rglob("*")) and any(p.is_file() for p in RESULTS_ROOT.rglob("*")):
            raise FileExistsError("Já há resultados locais. Use uma cópia limpa para uma nova sessão.")
        SESSION_ROOT.mkdir(parents=True)
        save_json(environment, existing)
        save_json(environment, RESULTS_ROOT / "environment.json")
    return environment

if PREPARE_YOLO:
    YOLO_RUNTIME = prepare_yolov5(FASE_ROOT / ".runtime/yolov5", install=INSTALL_YOLO_DEPENDENCIES)
if RUN_PREFLIGHT:
    PREFLIGHT = preflight_colab()
    if RESTORE_RESULTS:
        custom_runs = recover_yolo_runs()
    print(json.dumps(PREFLIGHT, ensure_ascii=False, indent=2))
if RUN_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
else:
    print("Preflight e treinos aguardam habilitação explícita no Colab.")


## 9.1. Experimento de 30 épocas

In [ ]:
START_FINAL_EXPERIMENTS = False  # altere somente depois de conferir o preflight
if START_FINAL_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
    RUN_EXPERIMENTS = True
    print("Experimentos finais habilitados para esta sessão validada.")


In [ ]:
if RUN_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
    if 30 not in custom_runs:
        params = CONFIG["yolo_custom"]
        custom_runs[30] = train_custom_yolo(YOLO_RUNTIME, DATASET_ROOT, root=FASE_ROOT, epochs=30, batch_size=params["batch_size"], image_size=params["image_size"], seed=42, output_dir=SESSION_ROOT / "runs/yolo_custom")
        persist_results()
    print(custom_runs[30]["metrics"])
else:
    print("Treinamento de 30 épocas pendente.")


## 9.2. Experimento de 60 épocas

In [ ]:
if RUN_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
    if 60 not in custom_runs:
        params = CONFIG["yolo_custom"]
        custom_runs[60] = train_custom_yolo(YOLO_RUNTIME, DATASET_ROOT, root=FASE_ROOT, epochs=60, batch_size=params["batch_size"], image_size=params["image_size"], seed=42, output_dir=SESSION_ROOT / "runs/yolo_custom")
        persist_results()
    print(custom_runs[60]["metrics"])
else:
    print("Treinamento de 60 épocas pendente.")


## 9.3. Comparação dos experimentos
Os resumos usam a época de melhor fitness do YOLOv5, correspondente a `best.pt`.
Selecione o experimento por mAP@0.5:0.95 de validação; desempate por mAP@0.5
e depois pelo menor tempo real de treinamento; em nova igualdade, menos épocas. O teste não participa dessa escolha.

> Esta seção será preenchida após a execução com o dataset real.


In [ ]:
if RUN_EXPERIMENTS:
    if set(custom_runs) != {30, 60}:
        raise RuntimeError("Conclua ou recupere os dois treinamentos antes da comparação.")
    summaries = [custom_runs[epoch]["metrics"] for epoch in (30, 60)]
    invariant_keys = ("configuration", "dataset_fingerprint", "runtime_sha")
    if any(summaries[0][key] != summaries[1][key] for key in invariant_keys):
        raise ValueError("Os experimentos diferem além de épocas/nome.")
    if any(s["environment"]["contract"] != PREFLIGHT["contract"] for s in summaries):
        raise ValueError("Experimentos foram executados em condições diferentes.")
    comparison = compare_yolo_epochs(summaries)
    save_table(comparison, RESULTS_ROOT / "comparison/yolo_epochs_comparison.csv")
    selected = select_custom_experiment(summaries)
    save_json({"experiment": selected["experiment"], "epochs": selected["epochs"], "metrics_split": "val", "validation_metrics": {key: selected[key] for key in ("map50_95", "map50", "precision", "recall", "best_epoch", "training_seconds")}, "criterion": "maior map50_95; maior map50; menor training_seconds; menos epochs; igualdade numérica sem tolerância adicional", "dataset_fingerprint": DATASET_FINGERPRINT, "checkpoint_sha256": selected["checkpoint_sha256"]}, RESULTS_ROOT / "yolo_custom/selected_model.json")
    from IPython.display import display, Image as DisplayImage
    display(comparison)
    for epoch, run in custom_runs.items():
        figure = plot_yolo_curves(run["run_dir"] / "results.csv", RESULTS_ROOT / f"yolo_custom/curves_{epoch}.png")
        display(DisplayImage(filename=str(figure)))
    persist_results()
    print("Selecionado exclusivamente pela validação:", selected["experiment"])
else:
    print("Comparação 30/60 pendente.")


## 10. Teste da YOLO customizada — Task 16
Avalie apenas o experimento selecionado pela validação. A inferência salva
as oito imagens processadas, incluindo ambas as classes e imagens sem
detecção. Depois da execução, discuta falsos positivos/negativos reais;
não é obrigatório existir um erro. Métricas e tempos têm escopos distintos.


In [ ]:
def show_image_grid(paths, titles=None, columns=4):
    import matplotlib.pyplot as plt
    from PIL import Image
    paths = [Path(p) for p in paths]
    if not paths or any(not p.is_file() for p in paths):
        raise ValueError("Grade exige imagens reais existentes.")
    rows = (len(paths) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(4*columns, 4*rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for i, (axis, path) in enumerate(zip(axes.flat, paths)):
        with Image.open(path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(titles[i] if titles else path.name, fontsize=9)
    figure.tight_layout(); plt.show(); plt.close(figure)


In [ ]:
if RUN_EXPERIMENTS:
    if selected is None:
        raise RuntimeError("Selecione pela validação antes do teste.")
    run = custom_runs[selected["epochs"]]
    directory = RESULTS_ROOT / "yolo_custom"
    metrics_file = directory / "test_metrics.json"
    if RESTORE_RESULTS and metrics_file.is_file() and (directory / "summary.json").is_file():
        custom_test = json.loads(metrics_file.read_text(encoding="utf-8"))
        custom_inference = load_saved_inference(directory)
    else:
        custom_test = validate_custom_yolo(YOLO_RUNTIME, run["weights"], DATASET_ROOT, root=FASE_ROOT, output_dir=SESSION_ROOT / "runs/yolo_evaluation")
        custom_inference = detect_with_custom_yolo(YOLO_RUNTIME, run["weights"], DATASET_ROOT, directory, image_size=640)
        custom_test.update(custom_inference["summary"], metrics_split="test", experiment=selected["experiment"], environment=PREFLIGHT, checkpoint_sha256=selected["checkpoint_sha256"])
        save_json(custom_test, metrics_file)
        persist_results()
    if custom_test.get("experiment") != selected["experiment"] or custom_test.get("checkpoint_sha256") != selected["checkpoint_sha256"] or custom_test.get("dataset_fingerprint") != DATASET_FINGERPRINT or custom_test.get("metrics_split") != "test":
        raise ValueError("Teste preservado não corresponde ao checkpoint selecionado e dataset atual.")
    print(custom_test)
    show_image_grid(custom_inference["images"])
else:
    print("Teste customizado pendente.")


## 11. YOLOv5 padrão COCO
A carga abaixo pode baixar os pesos oficiais. Os IDs são resolvidos pelos
nomes `bottle` e `backpack`. A inferência utiliza exatamente `test/images`.
As tabelas registram também imagens sem detecções. Confiança não é mAP;
esta abordagem terá a métrica quantitativa principal indisponível enquanto
não houver uma avaliação de detecção com remapeamento COCO dos labels.

In [ ]:
if RUN_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
    directory = RESULTS_ROOT / "yolo_standard"
    if RESTORE_RESULTS and (directory / "summary.json").is_file():
        standard_inference = load_saved_inference(directory)
    else:
        standard_model = load_standard_yolo(YOLO_RUNTIME, CONFIG["yolo_standard"]["model"])
        standard_inference = infer_folder(standard_model, DATASET_ROOT / "test/images", output_dir=directory, image_size=640, warmup=1)
        del standard_model
        import torch
        gc.collect(); torch.cuda.empty_cache()
        persist_results()
    print(standard_inference["summary"])
    display(standard_inference["detections"])
    show_image_grid(standard_inference["images"])
else:
    print("Inferência COCO pendente.")


## 12. CNN treinada do zero
### 12.1. Preparação da classificação
Uma imagem deve ter apenas um ID distinto nos labels YOLO; múltiplos objetos
da mesma classe são aceitos. Labels ausentes, vazios ou com ambas as classes
impedem o manifesto. As três abordagens preservam o mesmo split de teste.

### Pipeline da CNN do zero

Construção, preparação de imagens, treinamento e avaliação. A CNN tem pesos aleatórios; nenhuma dessas funções treina ao ser definida.

In [ ]:
"""Small CNN with random weights; TensorFlow is imported only when needed."""
from pathlib import Path
import json
from time import perf_counter

def build_model(image_size=(224, 224), seed=42):
    import tensorflow as tf
    apply_seed(seed, tensorflow=True)
    inputs = tf.keras.Input(shape=(*image_size, 3), name="normalized_rgb")
    augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal", seed=seed), tf.keras.layers.RandomRotation(.05, seed=seed + 1), tf.keras.layers.RandomZoom(.1, seed=seed + 2)], name="train_only_augmentation")
    # Keras random preprocessing layers are inactive during inference/evaluation.
    x = augmentation(inputs)
    for filters in (16, 32, 64):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(.3, seed=seed)(x)
    outputs = tf.keras.layers.Dense(2, activation="softmax", name="class_probabilities")(x)
    model = tf.keras.Model(inputs, outputs, name="fase6_cnn_from_scratch")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def make_dataset(manifest, split, image_size=(224, 224), batch_size=8, seed=42):
    import tensorflow as tf
    if split not in ("train", "val", "test"):
        raise ValueError(f"Split inválido: {split}")
    rows = manifest[manifest["split"] == split]
    if rows.empty:
        raise ValueError(f"Dataset ainda não integrado: split {split} vazio.")
    paths = rows.image_path.astype(str).tolist()
    ids = rows.class_id.astype(int).tolist()
    if any(cid not in CLASSES for cid in ids):
        raise ValueError("Classe inválida no manifesto.")
    dataset = tf.data.Dataset.from_tensor_slices((paths, ids))
    if split == "train":
        dataset = dataset.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
    def decode(path, class_id):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(tf.cast(image, tf.float32), image_size) / 255.0
        return image, class_id
    return dataset.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def train_cnn(dataset_root, root=None, image_size=(224, 224), batch_size=8, epochs=30, seed=42, run_dir=None):
    require_experiment_preflight(dataset_root)
    import tensorflow as tf
    root = Path(root).resolve() if root else find_root()
    manifest = classification_manifest(dataset_root)
    train = make_dataset(manifest, "train", image_size, batch_size, seed)
    val = make_dataset(manifest, "val", image_size, batch_size, seed)
    run_dir = Path(run_dir or root / "runs/cnn")
    if run_dir.exists():
        raise FileExistsError(f"Treinamento CNN já existe: {run_dir}")
    run_dir.mkdir(parents=True)
    model = build_model(image_size, seed)
    weights = run_dir / "best.keras"
    callback = tf.keras.callbacks.ModelCheckpoint(str(weights), monitor="val_loss", save_best_only=True)
    start = perf_counter()
    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=[callback])
    duration = perf_counter() - start
    result_dir = root / "results/cnn"
    result_dir.mkdir(parents=True, exist_ok=True)
    history_data = {key: [float(x) for x in values] for key, values in history.history.items()}
    (result_dir / "history.json").write_text(json.dumps(history_data, indent=2, allow_nan=False), encoding="utf-8")
    completed = len(history_data["loss"])
    if completed != epochs:
        raise ValueError(f"CNN incompleta: {completed}/{epochs} épocas.")
    best_epoch = min(range(completed), key=lambda i: history_data["val_loss"][i]) + 1
    save_json({"training_seconds": duration, "epochs": epochs, "completed_epochs": completed, "best_epoch": best_epoch, "seed": seed, "image_size": list(image_size), "batch_size": batch_size, "dataset_fingerprint": dataset_fingerprint(dataset_root), "checkpoint_sha256": file_sha256(weights), "environment": PREFLIGHT}, result_dir / "training.json")
    manifest.to_csv(run_dir / "classification_manifest.csv", index=False)
    return {"best_epoch": best_epoch, "model": tf.keras.models.load_model(weights), "weights": weights, "history": history_data, "training_seconds": duration, "manifest": manifest}

def evaluate_cnn(model, manifest, image_size=(224, 224), warmup=1, output_dir=None):
    import numpy as np
    import tensorflow as tf
    from PIL import Image
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
    rows = manifest[manifest["split"] == "test"]
    if rows.empty:
        raise ValueError("Dataset ainda não integrado: teste CNN vazio.")
    if warmup < 0:
        raise ValueError("warmup deve ser não negativo.")
    def predict(array):
        tensor = tf.image.resize(tf.cast(array, tf.float32), image_size) / 255.0
        return model(tf.expand_dims(tensor, 0), training=False).numpy()[0]
    with Image.open(rows.image_path.iloc[0]) as image:
        first = np.array(image.convert("RGB"))
    for _ in range(warmup):
        predict(first)
    predicted, times, records = [], [], []
    for row in rows.itertuples():
        with Image.open(row.image_path) as image:
            array = np.array(image.convert("RGB"))
        probabilities, duration = measure_call(predict, array)
        cid = int(np.argmax(probabilities))
        predicted.append(cid)
        times.append(duration)
        records.append({"image": Path(row.image_path).name, "true_class": row.class_name, "predicted_class": CLASSES[cid], "confidence": float(probabilities[cid]), "inference_ms": duration})
    truth = rows.class_id.astype(int).tolist()
    precision, recall, f1, _ = precision_recall_fscore_support(truth, predicted, labels=[0, 1], average="macro", zero_division=0)
    metrics = {"accuracy": float(accuracy_score(truth, predicted)), "precision": float(precision), "recall": float(recall), "f1": float(f1), "averaging": "macro", "confusion_matrix": confusion_matrix(truth, predicted, labels=[0, 1]).tolist(), "class_names": list(CLASSES.values()), "metrics_split": "test", "dataset_fingerprint": dataset_fingerprint(DATASET_ROOT), "environment": PREFLIGHT, **summarize_times(times), "warmup": warmup, "timing_scope": "preprocessing and model call; excludes image loading, model loading, warm-up and saving"}
    if output_dir:
        import pandas as pd
        directory = Path(output_dir)
        directory.mkdir(parents=True, exist_ok=True)
        (directory / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")
        pd.DataFrame(records).to_csv(directory / "predictions.csv", index=False)
    return metrics


In [ ]:
if DATASET_READY:
    manifest = classification_manifest(DATASET_ROOT)
    print(manifest.groupby(["split", "class_name"]).size())
else:
    print("Manifesto de classificação aguarda dataset real.")

### 12.2. Arquitetura
Três blocos Conv2D/ReLU/MaxPooling com 16, 32 e 64 filtros, GlobalAveragePooling,
Dense de 64 unidades, dropout 0.3 e saída softmax de duas classes.
Loss: sparse categorical crossentropy. Sem backbone ou pesos pré-treinados.
Imagens são redimensionadas para 224 × 224 e normalizadas para [0,1].
Flip horizontal, rotação pequena e zoom moderado atuam somente no treino.
O smoke test sintético abaixo verifica shape, sem produzir resultado acadêmico.

In [ ]:
if RUN_CNN_SMOKE_TEST:
    import tensorflow as tf
    cnn_model = build_model(tuple(CONFIG["cnn"]["image_size"]), 42)
    cnn_model.summary()
    synthetic_output = cnn_model(tf.random.uniform((1, 224, 224, 3)), training=False)
    assert synthetic_output.shape == (1, 2)
    print("Smoke test técnico: saída", synthetic_output.shape)
    del cnn_model
    tf.keras.backend.clear_session()
else:
    print("Smoke test sintético opcional desativado; não produz métrica acadêmica.")


### 12.3. Treinamento
Treine por 30 épocas e selecione o checkpoint pela menor loss de validação.
Pesos `.keras` ficam em `runs/`, ignorado pelo Git; somente histórico e duração
consolidados são salvos em `results/cnn`.

In [ ]:
if RUN_EXPERIMENTS:
    require_experiment_preflight(DATASET_ROOT)
    params = CONFIG["cnn"]
    run_dir = SESSION_ROOT / "runs/cnn"
    training_file = RESULTS_ROOT / "cnn/training.json"
    if RESTORE_RESULTS and training_file.is_file():
        import tensorflow as tf
        training = json.loads(training_file.read_text(encoding="utf-8"))
        weights = run_dir / "best.keras"
        if training["completed_epochs"] != 30 or training["dataset_fingerprint"] != DATASET_FINGERPRINT or file_sha256(weights) != training["checkpoint_sha256"]:
            raise ValueError("Checkpoint CNN incompleto ou divergente.")
        cnn_run = {"model": tf.keras.models.load_model(weights), "weights": weights, "history": json.loads((RESULTS_ROOT / "cnn/history.json").read_text(encoding="utf-8")), "training_seconds": training["training_seconds"], "best_epoch": training["best_epoch"], "manifest": classification_manifest(DATASET_ROOT)}
    else:
        cnn_run = train_cnn(DATASET_ROOT, root=FASE_ROOT, image_size=tuple(params["image_size"]), batch_size=8, epochs=30, seed=42, run_dir=run_dir)
    for figure in plot_cnn_history(cnn_run["history"], RESULTS_ROOT / "cnn"):
        display(DisplayImage(filename=str(figure)))
    persist_results()
else:
    print("Treinamento CNN pendente.")


### 12.4. Avaliação
Accuracy mede a proporção de classes corretas; precision, recall e F1 macro
atribuem o mesmo peso a cada classe. A matriz de confusão evidencia os erros
entre bottle e backpack. Avalie o melhor checkpoint somente no teste.

In [ ]:
if RUN_EXPERIMENTS:
    if cnn_run is None:
        raise RuntimeError("Treine ou recupere o melhor checkpoint CNN antes do teste.")
    metrics_file = RESULTS_ROOT / "cnn/metrics.json"
    if RESTORE_RESULTS and metrics_file.is_file():
        cnn_metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
        if cnn_metrics["dataset_fingerprint"] != DATASET_FINGERPRINT or cnn_metrics.get("checkpoint_sha256") != file_sha256(cnn_run["weights"]):
            raise ValueError("Métricas CNN de outro dataset.")
    else:
        cnn_metrics = evaluate_cnn(cnn_run["model"], cnn_run["manifest"], image_size=(224, 224), output_dir=RESULTS_ROOT / "cnn", warmup=1)
    cnn_metrics["checkpoint_sha256"] = file_sha256(cnn_run["weights"])
    save_json(cnn_metrics, metrics_file)
    figure = plot_confusion_matrix(cnn_metrics, RESULTS_ROOT / "cnn/confusion_matrix.png")
    display(DisplayImage(filename=str(figure)))
    predictions = pd.read_csv(RESULTS_ROOT / "cnn/predictions.csv")
    if set(predictions.image) != {p.name for p in list_images(DATASET_ROOT / "test/images")} or len(predictions) != 8:
        raise ValueError("Predições CNN não correspondem ao mesmo teste.")
    titles = [f"Real: {row.true_class} | Prevista: {row.predicted_class}\nConfiança: {row.confidence:.3f}" for row in predictions.itertuples()]
    show_image_grid([DATASET_ROOT / "test/images" / name for name in predictions.image], titles)
    print(cnn_metrics)
    persist_results()
else:
    print("Avaliação CNN pendente.")


## 13. Comparação geral
Detecção localiza objetos; classificação atribui uma classe à imagem inteira.
Não ordene CNN e YOLO como se accuracy e mAP fossem a mesma grandeza.
Compare tempos, facilidade de uso e limitações transversalmente; interprete
cada métrica dentro da sua tarefa. YOLO padrão não tem treino nesta base,
portanto seu tempo de treino é indisponível, e não um tempo medido igual a zero.

Os cronômetros usam `perf_counter`, excluem carga de modelo, leitura de arquivos,
warm-up e salvamento; incluem pré-processamento e chamada do modelo (NMS na YOLO).
YOLO sincroniza CUDA; CNN materializa a saída. Registre hardware, dispositivo,
versões e tamanhos diferentes de entrada. São tempos dos pipelines, não FLOPs.

> Esta seção será preenchida após a execução com o dataset real.

In [ ]:
if RUN_EXPERIMENTS:
    if not all(x is not None for x in (selected, custom_test, custom_inference, standard_inference, cnn_run, cnn_metrics)):
        raise RuntimeError("Benchmark exige resultados reais das três abordagens.")
    expected = {p.name for p in list_images(DATASET_ROOT / "test/images")}
    for inference in (custom_inference, standard_inference):
        if set(inference["timings"].image) != expected or len(inference["timings"]) != 8:
            raise ValueError("Abordagens não utilizaram o mesmo teste.")
    def timing_fields(metrics):
        return {key: metrics[key] for key in ("mean_inference_ms", "median_inference_ms", "std_inference_ms")}
    records = [
        {"approach": "yolo_custom", "task_type": "detection", "training_seconds": selected["training_seconds"], **timing_fields(custom_inference["summary"]), "primary_metric": "map50_95", "primary_metric_value": custom_test["map50_95"], **{k: custom_test[k] for k in ("precision", "recall", "map50", "map50_95")}, "f1": None, "notes": f"{selected['epochs']} épocas; seleção em val; teste reservado; imagem 640; integração exige labels e treino"},
        {"approach": "yolo_standard", "task_type": "detection", "training_seconds": None, **timing_fields(standard_inference["summary"]), "primary_metric": None, "primary_metric_value": None, "precision": None, "recall": None, "f1": None, "map50": None, "map50_95": None, "notes": "COCO; sem treino nesta base; mAP não calculado; imagem 640; uso direto com filtro por nomes"},
        {"approach": "cnn", "task_type": "classification", "training_seconds": cnn_run["training_seconds"], **timing_fields(cnn_metrics), "primary_metric": "accuracy", "primary_metric_value": cnn_metrics["accuracy"], **{k: cnn_metrics[k] for k in ("precision", "recall", "f1")}, "map50": None, "map50_95": None, "notes": "Do zero; métricas macro; teste comum; 224x224; requer classe única e pipeline próprio"},
    ]
    benchmark = build_benchmark(records)
    save_table(benchmark, RESULTS_ROOT / "comparison/final_benchmark.csv")
    figure = plot_times(benchmark, RESULTS_ROOT / "comparison/inference_times.png")
    display(benchmark); display(DisplayImage(filename=str(figure)))
    persist_results()
else:
    print("Benchmark final aguarda execuções reais das três abordagens.")


## 14. Conclusões — preencher após os resultados reais
- Qual foi a diferença de mAP, precision/recall e duração entre 30/60 épocas?
- As curvas de loss treino/validação sustentam ganho, saturação ou overfitting?
- Como a YOLO selecionada se comportou nas oito imagens de teste?
- Quais acertos/falhas foram observados na COCO, sem usar confiança como mAP?
- O que accuracy, F1 macro e matriz de confusão mostram sobre a CNN?
- Quais custos e dificuldades de integração foram observados em cada fluxo?

## 15. Pontos fortes
Rotulação revisada no Make Sense com proveniência, separação 32/4/4,
seleção pela validação e preservação dos artefatos. O desempenho dos modelos
permanece pendente até a execução real.

## 16. Limitações
Teste de apenas oito imagens. A anotação segue a classe de foco por imagem;
objetos incidentais da outra classe não foram anotados exaustivamente.
Há uma representação impressa de garrafa e uma candidata a mochila com
rodinhas, registradas na validação do Make Sense. A revisão foi realizada
por Codex, sem atribuir uma revisão humana independente ao aluno.
Accuracy e mAP não representam a mesma tarefa. Os tempos refletem pipelines
com entradas diferentes, não uma comparação isolada de operações.
Treinamentos, conclusões finais e vídeo ainda não foram realizados.
ESP32-CAM, Transfer Learning, Fine Tuning e segmentação ficam fora do escopo.

## 17. Devolução para consolidação
Ao terminar, execute a exportação abaixo e baixe este notebook no Colab
por **Arquivo > Fazer download > Fazer download do .ipynb**. Envie o `.ipynb`
executado e o ZIP dos resultados. Checkpoints e runs permanecem no Drive.
Não remova tabelas, gráficos ou detecções reais do notebook acadêmico.
O roteiro do vídeo e a auditoria final só receberão números após conferência.


In [ ]:
def export_small_results(results_root, output):
    root, output = Path(results_root), Path(output)
    required = ["environment.json", "yolo_custom/metrics_30_epochs.json", "yolo_custom/metrics_60_epochs.json", "yolo_custom/selected_model.json", "yolo_custom/test_metrics.json", "yolo_custom/history_30_epochs.csv", "yolo_custom/history_60_epochs.csv", "yolo_custom/curves_30.png", "yolo_custom/curves_60.png", "yolo_custom/detections.csv", "yolo_custom/inference_times.csv", "yolo_custom/summary.json", "yolo_standard/detections.csv", "yolo_standard/inference_times.csv", "yolo_standard/summary.json", "cnn/history.json", "cnn/training.json", "cnn/metrics.json", "cnn/predictions.csv", "cnn/cnn_loss.png", "cnn/cnn_accuracy.png", "cnn/confusion_matrix.png", "comparison/yolo_epochs_comparison.csv", "comparison/final_benchmark.csv", "comparison/inference_times.png"]
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise ValueError(f"Resultados incompletos; não exportar como entrega final: {missing}")
    expected = {f"evidence_{p.stem}.jpg" for p in list_images(DATASET_ROOT / "test/images")}
    for approach in ("yolo_custom", "yolo_standard"):
        if {p.name for p in (root / approach / "detections").glob("*.jpg")} != expected:
            raise ValueError("Exporte as oito imagens processadas de cada YOLO.")
    files = [p for p in sorted(root.rglob("*")) if p.is_file() and p.suffix.lower() in {".json", ".csv", ".png", ".jpg"}]
    if any(p.is_symlink() or p.stat().st_size > 10*1024*1024 for p in files):
        raise ValueError("Resultados contêm link simbólico ou arquivo pesado inesperado.")
    if output.exists():
        raise FileExistsError("ZIP de devolução já existe; preserve-o ou escolha outro nome.")
    output.parent.mkdir(parents=True, exist_ok=True)
    manifest = {"session": SESSION_NAME, "dataset_fingerprint": DATASET_FINGERPRINT, "files": [{"path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size, "sha256": file_sha256(p)} for p in files]}
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, "results/" + path.relative_to(root).as_posix())
        archive.writestr("return_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    with zipfile.ZipFile(output) as archive:
        if archive.testzip() is not None:
            raise ValueError("Falha CRC na exportação.")
    return output

if EXPORT_RESULTS:
    require_experiment_preflight(DATASET_ROOT)
    persist_results()
    return_zip = export_small_results(RESULTS_ROOT, SESSION_ROOT / f"fase6_resultados_{SESSION_NAME}.zip")
    print("Resultados reais preservados em:", return_zip)
    if DOWNLOAD_RESULTS:
        from google.colab import files
        files.download(str(return_zip))
else:
    print("Exportação aguarda os resultados reais e EXPORT_RESULTS=True.")
